# US-Iran War 2026: Crude Oil Crisis & Claude AI Intelligence Dashboard

**Techniques:** Deep Learning (LSTM) · RAG · GenAI · NLP · Geospatial AI · Monte Carlo Simulation

**Data sources:** BBC News, Al Jazeera, Reuters, Washington Post — March 2026

---
## Project Overview

This notebook analyses how the US-Israel military operation against Iran (Operation Epic Fury, launched March 3 2026) caused one of the most volatile oil price episodes in modern history — and how Anthropic's Claude AI was embedded in Palantir's Maven Smart System to support targeting decisions.

### Modules
1. Oil price timeline analysis and visualisation
2. Global economic impact scoring (NLP-based)
3. LSTM-based price forecasting
4. Monte Carlo scenario simulation (Ceasefire / Escalation / Hormuz Blocked)
5. Claude AI military deployment timeline
6. RAG-powered Q&A with Claude API

In [ ]:
# Install dependencies
!pip install anthropic pandas numpy matplotlib seaborn plotly scikit-learn tensorflow -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Dark theme
plt.style.use('dark_background')
AMBER = '#f59e0b'
RED   = '#ef4444'
GREEN = '#22c55e'
BLUE  = '#38bdf8'
PURPLE= '#a78bfa'

print('Libraries loaded successfully.')

---
## 1. Load Datasets

In [ ]:
# Load all datasets
oil_df      = pd.read_csv('oil_price_timeline.csv', parse_dates=['date'])
world_df    = pd.read_csv('world_impact.csv')
scenario_df = pd.read_csv('ml_scenario_forecasts.csv')
claude_df   = pd.read_csv('claude_military_timeline.csv')

print('=== Oil Price Timeline ===')
print(oil_df.to_string(index=False))
print(f'\nPrice range: ${oil_df.brent_crude.min()} - ${oil_df.brent_crude.max()}')
print(f'Total surge: {((oil_df.brent_crude.max() - oil_df.brent_crude.iloc[0]) / oil_df.brent_crude.iloc[0] * 100):.1f}%')

---
## 2. Oil Price Timeline Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#060810')
ax.set_facecolor('#060810')

x = range(len(oil_df))
ax.fill_between(x, oil_df['brent_crude'], alpha=0.15, color=AMBER)
ax.fill_between(x, oil_df['wti'],         alpha=0.10, color=BLUE)
ax.plot(x, oil_df['brent_crude'], color=AMBER, linewidth=2.5, label='Brent Crude', marker='o', markersize=5)
ax.plot(x, oil_df['wti'],         color=BLUE,  linewidth=2,   label='WTI',         marker='o', markersize=4)
ax.axhline(y=100, color=RED, linestyle='--', alpha=0.5, label='$100 threshold')

# Annotate events
events = oil_df[oil_df['event'].notna()]
for idx, row in events.iterrows():
    xi = list(oil_df.index).index(idx)
    ax.annotate(
        row['event'],
        xy=(xi, row['brent_crude']),
        xytext=(xi, row['brent_crude'] + 6),
        fontsize=7.5, color='white', ha='center',
        arrowprops=dict(arrowstyle='->', color=AMBER, lw=1),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#1e1e2e', edgecolor=AMBER, alpha=0.8)
    )

ax.set_xticks(x)
ax.set_xticklabels(oil_df['date'].dt.strftime('%b %d'), rotation=30, color='#94a3b8', fontsize=9)
ax.set_ylabel('Price (USD/barrel)', color='#94a3b8')
ax.set_title('Brent Crude & WTI — US-Iran War Price Spike (Feb-Mar 2026)', 
             color='white', fontsize=13, fontweight='bold', pad=15)
ax.tick_params(colors='#94a3b8')
ax.spines[:].set_color('#1e293b')
ax.legend(facecolor='#0a0c14', edgecolor='#1e293b', labelcolor='white', fontsize=9)
ax.grid(alpha=0.06)
plt.tight_layout()
plt.savefig('oil_price_timeline.png', dpi=150, bbox_inches='tight', facecolor='#060810')
plt.show()
print('Chart saved: oil_price_timeline.png')

---
## 3. Descriptive Statistics & Price Volatility

In [ ]:
# Calculate daily returns and volatility
oil_df['brent_return'] = oil_df['brent_crude'].pct_change() * 100
oil_df['wti_return']   = oil_df['wti'].pct_change() * 100

print('=== Brent Crude Statistics ===')
print(oil_df['brent_crude'].describe().round(2))

print('\n=== Daily Return Statistics (%) ===')
print(oil_df['brent_return'].describe().round(2))

print(f'\nPre-war price (Feb 20):  ${oil_df.brent_crude.iloc[0]}')
print(f'Peak price    (Mar 8):   ${oil_df.brent_crude.max()}')
print(f'Current price (Mar 10):  ${oil_df.brent_crude.iloc[-1]}')
print(f'Peak-to-current drop:    {((oil_df.brent_crude.max() - oil_df.brent_crude.iloc[-1]) / oil_df.brent_crude.max() * 100):.1f}%')
print(f'Still above pre-war by:  {((oil_df.brent_crude.iloc[-1] - oil_df.brent_crude.iloc[0]) / oil_df.brent_crude.iloc[0] * 100):.1f}%')

In [ ]:
# Daily returns bar chart
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#060810')
ax.set_facecolor('#060810')

returns = oil_df['brent_return'].dropna()
colors  = [GREEN if r < 0 else RED for r in returns]
ax.bar(range(len(returns)), returns, color=colors, alpha=0.85)
ax.axhline(0, color='#475569', linewidth=0.8)
ax.set_xticks(range(len(returns)))
ax.set_xticklabels(oil_df['date'].dt.strftime('%b %d').iloc[1:], rotation=30, color='#94a3b8', fontsize=9)
ax.set_ylabel('Daily Return (%)', color='#94a3b8')
ax.set_title('Brent Crude Daily Returns — Volatility During Iran War', color='white', fontsize=12, fontweight='bold')
ax.tick_params(colors='#94a3b8')
ax.spines[:].set_color('#1e293b')
ax.grid(alpha=0.05)
plt.tight_layout()
plt.savefig('daily_returns.png', dpi=150, bbox_inches='tight', facecolor='#060810')
plt.show()

---
## 4. LSTM Price Forecasting

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# Prepare data
prices = oil_df['brent_crude'].values.reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(prices)

# Create sequences (window=3 given limited data)
def create_sequences(data, window=3):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled)
print(f'Sequences created: X shape {X.shape}, y shape {y.shape}')

# Build LSTM model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(3, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
history = model.fit(X, y, epochs=200, batch_size=4, verbose=0, validation_split=0.2)

# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#060810')
ax.set_facecolor('#060810')
ax.plot(history.history['loss'],     color=AMBER, label='Train Loss')
ax.plot(history.history['val_loss'], color=BLUE,  label='Val Loss', linestyle='--')
ax.set_title('LSTM Training Loss', color='white', fontweight='bold')
ax.set_xlabel('Epoch', color='#94a3b8')
ax.set_ylabel('MSE Loss', color='#94a3b8')
ax.tick_params(colors='#94a3b8')
ax.spines[:].set_color('#1e293b')
ax.legend(facecolor='#0a0c14', edgecolor='#1e293b', labelcolor='white')
ax.grid(alpha=0.05)
plt.tight_layout()
plt.savefig('lstm_training.png', dpi=150, bbox_inches='tight', facecolor='#060810')
plt.show()

In [ ]:
# Predict on training data and forecast next 4 weeks
train_pred = scaler.inverse_transform(model.predict(X, verbose=0))

# Forecast future using last known window
last_window = scaled[-3:].reshape(1, 3, 1)
future_preds = []
for _ in range(4):
    pred = model.predict(last_window, verbose=0)[0][0]
    future_preds.append(pred)
    last_window = np.append(last_window[:, 1:, :], [[[pred]]], axis=1)

future_prices = scaler.inverse_transform(np.array(future_preds).reshape(-1, 1)).flatten()
print('4-Week LSTM Forecast (Brent Crude USD/barrel):')
for i, p in enumerate(future_prices, 1):
    print(f'  W+{i}: ${p:.2f}')

---
## 5. Monte Carlo Scenario Simulation

In [ ]:
np.random.seed(42)
n_simulations = 1000
n_weeks       = 4
current_price = 93.0

scenarios = {
    'Ceasefire':      {'drift': -0.055, 'vol': 0.04,  'color': GREEN},
    'Escalation':     {'drift':  0.040, 'vol': 0.06,  'color': AMBER},
    'Hormuz Blocked': {'drift':  0.120, 'vol': 0.10,  'color': RED},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.patch.set_facecolor('#060810')
fig.suptitle('Monte Carlo Oil Price Simulation — 1,000 Paths per Scenario', 
             color='white', fontsize=13, fontweight='bold', y=1.01)

results = {}
for ax, (name, params) in zip(axes, scenarios.items()):
    ax.set_facecolor('#060810')
    paths = np.zeros((n_simulations, n_weeks + 1))
    paths[:, 0] = current_price
    for t in range(1, n_weeks + 1):
        shocks = np.random.normal(params['drift'], params['vol'], n_simulations)
        paths[:, t] = paths[:, t-1] * (1 + shocks)

    results[name] = paths
    # Plot paths
    for path in paths[:200]:
        ax.plot(path, color=params['color'], alpha=0.04, linewidth=0.8)
    # Plot mean
    mean_path = paths.mean(axis=0)
    ax.plot(mean_path, color=params['color'], linewidth=2.5, label='Mean path')
    # Confidence interval
    p5  = np.percentile(paths, 5,  axis=0)
    p95 = np.percentile(paths, 95, axis=0)
    ax.fill_between(range(n_weeks+1), p5, p95, alpha=0.15, color=params['color'])

    final_mean = mean_path[-1]
    ax.set_title(f'{name}\nMean W+4: ${final_mean:.0f}', color='white', fontsize=10, fontweight='bold')
    ax.set_xlabel('Weeks', color='#94a3b8', fontsize=9)
    ax.set_ylabel('Brent (USD/barrel)', color='#94a3b8', fontsize=9)
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#1e293b')
    ax.grid(alpha=0.05)
    ax.set_xticks(range(n_weeks+1))
    ax.set_xticklabels(['Now', 'W+1', 'W+2', 'W+3', 'W+4'], fontsize=8)

plt.tight_layout()
plt.savefig('monte_carlo_scenarios.png', dpi=150, bbox_inches='tight', facecolor='#060810')
plt.show()

# Summary stats
print('\n=== Monte Carlo Summary (W+4) ===')
for name, paths in results.items():
    final = paths[:, -1]
    print(f'{name:20s} | Mean: ${final.mean():.0f} | P5: ${np.percentile(final,5):.0f} | P95: ${np.percentile(final,95):.0f}')

---
## 6. Global Economic Impact Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#060810')
ax.set_facecolor('#060810')

world_sorted = world_df.sort_values('impact_score', ascending=True)
color_map = {'importer': RED, 'exporter': GREEN, 'mixed': AMBER}
bar_colors = [color_map[t] for t in world_sorted['type']]

bars = ax.barh(world_sorted['country'], world_sorted['impact_score'], 
               color=bar_colors, alpha=0.85, height=0.65)

for bar, score in zip(bars, world_sorted['impact_score']):
    ax.text(score + 1, bar.get_y() + bar.get_height()/2, 
            str(score), va='center', color='#94a3b8', fontsize=9)

legend_handles = [
    mpatches.Patch(color=RED,   label='Net Importer — highest pain'),
    mpatches.Patch(color=GREEN, label='Net Exporter — revenue windfall'),
    mpatches.Patch(color=AMBER, label='Mixed exposure'),
]
ax.legend(handles=legend_handles, facecolor='#0a0c14', edgecolor='#1e293b', 
          labelcolor='white', fontsize=9, loc='lower right')

ax.set_xlabel('Impact Score (0-100)', color='#94a3b8')
ax.set_title('Global Economic Impact Index — US-Iran War Oil Shock', 
             color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='#94a3b8')
ax.spines[:].set_color('#1e293b')
ax.grid(axis='x', alpha=0.05)
plt.tight_layout()
plt.savefig('world_impact.png', dpi=150, bbox_inches='tight', facecolor='#060810')
plt.show()

In [ ]:
# Correlation analysis — oil price vs impact score
print('=== Impact Score Statistics by Country Type ===')
print(world_df.groupby('type')['impact_score'].describe().round(2))

print('\n=== Top 5 Most Impacted Countries ===')
print(world_df.nlargest(5, 'impact_score')[['country','impact_score','type','market_detail']].to_string(index=False))

---
## 7. Claude AI Military Deployment Timeline

In [ ]:
print('=== Claude AI Battlefield Deployment — Operation Epic Fury ===')
print('=' * 65)
for _, row in claude_df.iterrows():
    print(f'\n[{row["phase"]}]  |  {row["date"]}')
    print(f'  Tech Stack: {row["tech_stack"]}')
    print(f'  {row["description"]}')
    print('-' * 65)

print('\nKey Facts:')
print('  - Claude embedded in Palantir Maven Smart System on classified networks')
print('  - ~1,000 targets generated on Day 1 with GPS + weapons recommendations')
print('  - Auto-generated legal justifications under international humanitarian law')
print('  - Trump EO banned Anthropic Feb 28 — OpenAI and xAI filled the gap')

---
## 8. RAG-Powered Q&A with Claude API

Set your `ANTHROPIC_API_KEY` as a Kaggle Secret or environment variable.

In [ ]:
import os
import anthropic

# Set API key — use Kaggle Secrets or os.environ
# from kaggle_secrets import UserSecretsClient
# os.environ['ANTHROPIC_API_KEY'] = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')

RAG_CONTEXT = """
KNOWLEDGE BASE — US-IRAN WAR OIL CRISIS 2026:

1. US-Iran War: The US and Israel launched Operation Epic Fury against Iran starting March 3, 2026.
   Mojtaba Khamenei named new supreme leader after strikes on Tehran.

2. Oil Prices: Brent crude rose from $74.20 (pre-war) to a peak of $119.80 on March 8 — a 61% surge.
   Fell to $93.05 on March 10 after Trump called it a 'short-term excursion'.
   Hormuz threat drove a 6% single-day drop.

3. Claude AI Military Use: Anthropic's Claude was embedded in Palantir's Maven Smart System on
   classified military networks. Generated ~1,000 prioritized targets on Day 1 using satellite imagery,
   SIGINT, and surveillance feeds. Auto-generated legal justifications for strikes. Trump banned
   Anthropic days before the war via executive order; OpenAI and xAI filled the gap.

4. Global Impact: Japan Nikkei +3.3%, South Korea Kospi +6.2% on ceasefire hopes. G7 discussed
   IEA reserve release. Bangladesh pump prices +22%, Philippines transport crisis.

5. Strait of Hormuz: ~20% of world oil passes through it. Trump threatened Iran 20x harder
   retaliation if blocked.

6. ML Scenarios (W+4 forecast): Ceasefire → $71/barrel. Escalation → $162/barrel.
   Hormuz closure → $241/barrel.
"""

def rag_query(question: str) -> str:
    client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
    message = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=1000,
        system=f"""You are an expert geopolitical AI analyst. Answer questions using ONLY the
        following knowledge base. Be precise and cite specific data points.\n\n{RAG_CONTEXT}""",
        messages=[{'role': 'user', 'content': question}]
    )
    return message.content[0].text

# Example queries
questions = [
    'How did Claude AI help the US military in Iran?',
    'What happens to oil prices if the Strait of Hormuz is blocked?',
    'Which countries were most economically impacted by the war?',
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {rag_query(q)}')
    print('-' * 70)

---
## 9. Summary Dashboard — Combined Visualisation

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Oil Price Timeline (Brent Crude)',
        'World Impact Score by Country',
        'ML Scenario Forecasts (W+4)',
        'Daily Price Volatility'
    ],
    vertical_spacing=0.14
)

# 1. Oil timeline
fig.add_trace(go.Scatter(
    x=oil_df['date'], y=oil_df['brent_crude'],
    name='Brent Crude', line=dict(color='#f59e0b', width=2.5),
    fill='tozeroy', fillcolor='rgba(245,158,11,0.08)'
), row=1, col=1)

# 2. World impact
color_map2 = {'importer': '#ef4444', 'exporter': '#22c55e', 'mixed': '#f59e0b'}
fig.add_trace(go.Bar(
    x=world_df['impact_score'], y=world_df['country'],
    orientation='h',
    marker_color=[color_map2[t] for t in world_df['type']],
    name='Impact Score'
), row=1, col=2)

# 3. Scenario forecasts
for col_name, color, label in [
    ('ceasefire_price', '#22c55e', 'Ceasefire'),
    ('escalation_price', '#f59e0b', 'Escalation'),
    ('hormuz_blocked_price', '#ef4444', 'Hormuz Blocked')
]:
    fig.add_trace(go.Scatter(
        x=scenario_df['week'], y=scenario_df[col_name],
        name=label, line=dict(color=color, width=2)
    ), row=2, col=1)

# 4. Daily returns
ret = oil_df['brent_return'].dropna()
fig.add_trace(go.Bar(
    x=oil_df['date'].iloc[1:], y=ret,
    marker_color=['#22c55e' if r < 0 else '#ef4444' for r in ret],
    name='Daily Return %'
), row=2, col=2)

fig.update_layout(
    height=700,
    title_text='US-Iran War 2026 — Oil Crisis & AI Intelligence Dashboard',
    title_font=dict(size=15, color='white'),
    paper_bgcolor='#060810',
    plot_bgcolor='#0a0c14',
    font=dict(color='#94a3b8'),
    showlegend=False
)
fig.update_xaxes(gridcolor='#1e293b')
fig.update_yaxes(gridcolor='#1e293b')
fig.write_html('summary_dashboard.html')
fig.show()
print('Interactive dashboard saved: summary_dashboard.html')

---
## 10. Key Findings

| Finding | Value |
|---|---|
| Total Brent price surge (pre-war to peak) | +61% ($74 → $120) |
| Peak price reached | $119.80 on March 8, 2026 |
| Single-day drop on Hormuz threat | -6% ($99 → $93) |
| Still above pre-war level | +25% as of March 10 |
| Most impacted country | Japan (score: 92) |
| Least impacted country | Nigeria (score: 30, exporter windfall) |
| Claude AI targets generated Day 1 | ~1,000 |
| Hormuz closure scenario forecast | $241/barrel (W+4) |
| Ceasefire scenario forecast | $71/barrel (W+4) |

---
*Data sources: BBC News, Al Jazeera, Reuters, Washington Post — March 2026*